# Spotify Hit Prediction - Preprocessing

L'objectif de cette étape est de préparer les données pour la modélisation en appliquant les transformations nécessaires:

- Nettoyage des données  
- Sélection des variables  
- Feature engineering (création et transformation de variables)  
- Encodage des variables catégorielles  
- Normalisation : mise à l’échelle des variables numériques  


In [1]:
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import warnings

warnings.filterwarnings("ignore")

Les données sont chargées depuis Hugging Face puis converties en DataFrame pandas pour les manipulations.

In [2]:
dataset = load_dataset("Faizasb/spotify-tracks-dataset")
df = dataset["train"].to_pandas()
print(f"Taille du dataset d'origine: {df.shape}")
df.head()

Taille du dataset d'origine: (114000, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## 1. Création de la variable cible

La variable `is_hit` est créée à partir de la variable `popularity`.  
Un morceau est considéré comme un hit s’il fait partie des 20% les plus populaires.

In [3]:
seuil = df["popularity"].quantile(0.8)
df["is_hit"] = df["popularity"] >= seuil
print(df["is_hit"].value_counts())

is_hit
False    90570
True     23430
Name: count, dtype: int64


## 2. Séparation des variables explicatives et de la cible

La variable cible `is_hit` est séparée des variables explicatives afin de préparer les données pour la modélisation.

In [4]:
X = df.drop("is_hit", axis=1)
y = df["is_hit"]

print(X.shape)
print(y.shape)

(114000, 21)
(114000,)


## 3. Séparation en jeu d'entraînement et de test (Train set et Test set)

Les données sont séparées en un jeu d'entraînement et un jeu de test afin d'évaluer les performances du modèle sur des données jamais vues.

La stratification est utilisée pour conserver la proportion de hits dans les deux jeux de données (environ 20% de hits dans chaque ensemble).

- Train set : 80% des données
- Test set : 20% des données

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (91200, 21)
Test: (22800, 21)


## 4. Sélection des variables

Certaines variables sont supprimées avant la modélisation :

- `Unnamed: 0` : index technique sans valeur informative.
- Variables d’identification (`track_id`, `track_name`, `album_name`) : elles n’apportent pas d’information utile.
- `artists` : risque de surapprentissage (le modèle pourrait mémoriser les artistes).
- `popularity` : utilisée pour créer la cible -->  risque de fuite d’information.
- `mode`, `key`, `time_signature` : variables peu informatives d’après l’EDA.

In [6]:
cols_to_drop = [
    "Unnamed: 0",
    "track_id",
    "track_name",
    "album_name",
    "artists",
    "popularity",
    "mode",
    "key",
    "time_signature",
]

X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (91200, 12)
Test: (22800, 12)


## 5. Traitement des valeurs manquantes et doublons

In [7]:
print("Nombre de valeurs manquantes par colonne dans X_train:")
print(X_train.isna().sum(), "\n")
print("Nombre de doublons: ", X_train.duplicated().sum())

Nombre de valeurs manquantes par colonne dans X_train:
duration_ms         0
explicit            0
danceability        0
energy              0
loudness            0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
track_genre         0
dtype: int64 

Nombre de doublons:  7976


In [8]:
print("Nombre de valeurs manquantes par colonne dans X_test:")
print(X_test.isna().sum(), "\n")
print("Nombre de doublons: ", X_test.duplicated().sum())

Nombre de valeurs manquantes par colonne dans X_test:
duration_ms         0
explicit            0
danceability        0
energy              0
loudness            0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
track_genre         0
dtype: int64 

Nombre de doublons:  876


In [9]:
print("Doublons dans X_train:", X_train.duplicated().mean().round(4) * 100, "%")
print("Doublons dans X_test:", X_test.duplicated().mean().round(4) * 100, "%")

Doublons dans X_train: 8.75 %
Doublons dans X_test: 3.84 %


Aucune valeur manquante n’est observée dans les données, ce qui signifie qu’aucun traitement spécifique n’est nécessaire à ce niveau.

On observe toutefois la présence de doublons dans les variables explicatives. Cela s’explique notamment par la suppression des variables d’identification (`track_id`, `Unnamed: 0`), qui permettaient de distinguer des morceaux potentiellement similaires.

Ces doublons correspondent en réalité à des morceaux partageant des caractéristiques très proches (voire identiques) et ne sont pas considérés comme des erreurs de données.

Par conséquent, ils sont conservés afin de ne pas perdre d’information.

## 6. Feature engineering : regroupement des genres

La variable `track_genre` contient un grand nombre de catégories (114), ce qui peut compliquer la modélisation.

Afin de réduire la complexité et améliorer la généralisation du modèle, les genres sont regroupés en grandes catégories musicales cohérentes.

In [10]:
def simplify_genre(genre):
    match genre:
        # ROCK
        case g if g in {
            "rock",
            "alt-rock",
            "alternative",
            "indie",
            "punk",
            "punk-rock",
            "grunge",
            "hard-rock",
            "emo",
            "goth",
            "psych-rock",
            "rock-n-roll",
            "rockabilly",
            "j-rock",
            "british",
        }:
            return "rock"

        # METAL
        case g if g in {
            "metal",
            "heavy-metal",
            "black-metal",
            "death-metal",
            "metalcore",
            "grindcore",
            "hardcore",
            "industrial",
        }:
            return "metal"

        # ELECTRONIC
        case g if g in {
            "electronic",
            "edm",
            "electro",
            "house",
            "deep-house",
            "chicago-house",
            "detroit-techno",
            "minimal-techno",
            "techno",
            "trance",
            "progressive-house",
            "hardstyle",
            "dubstep",
            "drum-and-bass",
            "breakbeat",
            "idm",
            "garage",
            "club",
            "dance",
            "disco",
            "trip-hop",
        }:
            return "electronic"

        # POP
        case g if g in {
            "pop",
            "synth-pop",
            "power-pop",
            "pop-film",
            "indie-pop",
            "k-pop",
            "j-pop",
            "j-idol",
            "j-dance",
            "cantopop",
            "mandopop",
        }:
            return "pop"

        # URBAN
        case g if g in {"hip-hop", "r-n-b"}:
            return "urban"

        # JAZZ / SOUL
        case g if g in {"jazz", "soul", "funk", "blues", "gospel", "groove"}:
            return "jazz_soul"

        # CLASSICAL
        case g if g in {"classical", "opera", "piano", "new-age"}:
            return "classical_instrumental"

        # FOLK / COUNTRY
        case g if g in {
            "folk",
            "country",
            "bluegrass",
            "honky-tonk",
            "acoustic",
            "singer-songwriter",
            "songwriter",
            "guitar",
        }:
            return "folk_country"

        # LATIN / WORLD
        case g if g in {
            "latin",
            "latino",
            "reggaeton",
            "salsa",
            "samba",
            "forro",
            "mpb",
            "pagode",
            "sertanejo",
            "brazil",
            "tango",
            "spanish",
            "french",
            "german",
            "swedish",
            "turkish",
            "indian",
            "iranian",
            "malay",
            "world-music",
            "afrobeat",
        }:
            return "latin_world"

        # REGGAE
        case g if g in {"reggae", "dancehall", "ska", "dub"}:
            return "reggae_caribbean"

        # AMBIENT / MOOD
        case g if g in {
            "ambient",
            "chill",
            "sleep",
            "study",
            "happy",
            "sad",
            "party",
            "romance",
        }:
            return "ambient_mood"

        # MEDIA / FUNCTIONAL
        case g if g in {"kids", "children", "anime", "disney", "comedy", "show-tunes"}:
            return "functional_media"

        # OTHER
        case _:
            return "other"

In [11]:
X_train["track_genre"] = X_train["track_genre"].apply(simplify_genre)
X_test["track_genre"] = X_test["track_genre"].apply(simplify_genre)

print(X_train["track_genre"].value_counts())
print()
print(
    f"On passe de {df['track_genre'].nunique()} genres musicaux à {X_train['track_genre'].nunique()} genres."
)

track_genre
latin_world               16823
electronic                16822
rock                      11951
pop                        8781
metal                      6408
ambient_mood               6368
folk_country               6337
functional_media           4852
jazz_soul                  4843
classical_instrumental     3219
reggae_caribbean           3157
urban                      1639
Name: count, dtype: int64

On passe de 114 genres musicaux à 12 genres.


## 7. Encodage des variables catégorielles

Les variables `track_genre` et `explicit` sont transformées en variables numériques à l’aide du One-Hot Encoding afin de pouvoir être utilisées par le modèle de machine learning.

In [12]:
categorical_cols = ["track_genre", "explicit"]

encoder = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")

Le One-Hot Encoding est appris uniquement sur le jeu d’entraînement afin d’éviter toute fuite d’information.

Les mêmes transformations sont ensuite appliquées au jeu de test.

In [13]:
encoder.fit(X_train[categorical_cols])
encoded_train = encoder.transform(X_train[categorical_cols])
encoded_test = encoder.transform(X_test[categorical_cols])

In [14]:
encoder

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",'first'
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",None
,"max_c

Les nouvelles colonnes générées par l’encodage sont récupérées afin de reconstruire un dataframe pandas lisible et exploitable.

In [15]:
encoded_cols = encoder.get_feature_names_out(categorical_cols)

encoded_train_df = pd.DataFrame(
    encoded_train, columns=encoded_cols, index=X_train.index
)

encoded_test_df = pd.DataFrame(encoded_test, columns=encoded_cols, index=X_test.index)

Les anciennes variables catégorielles sont supprimées puis remplacées par les nouvelles variables encodées.

In [16]:
X_train = X_train.drop(columns=categorical_cols)
X_test = X_test.drop(columns=categorical_cols)

X_train = pd.concat([X_train, encoded_train_df], axis=1)
X_test = pd.concat([X_test, encoded_test_df], axis=1)

In [17]:
print("Taille du X_train:", X_train.shape)
X_train.head()

Taille du X_train: (91200, 22)


,duration_ms,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,...,track_genre_folk_country,track_genre_functional_media,track_genre_jazz_soul,track_genre_latin_world,track_genre_metal,track_genre_pop,track_genre_reggae_caribbean,track_genre_rock,track_genre_urban,explicit_True
95669,169933,0.600,0.563,-10.534,0.0637,0.77300,0.000006,0.201,0.910,92.045,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
50489,452613,0.436,0.954,-6.315,0.0724,0.00242,0.258000,0.918,0.190,122.036,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
80571,272453,0.815,0.421,-8.125,0.0352,0.49100,0.000004,0.117,0.543,140.010,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
63080,237106,0.593,0.957,-2.012,0.0844,0.00559,0.000001,0.597,0.611,99.971,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
92590,150866,0.407,0.826,-8.290,0.0495,0.69300,0.000016,0.285,0.891,192.210,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
